# Data Validation & Test Cases for Retail Warehouse

This notebook contains comprehensive validation queries to ensure data quality and correctness across the Gold layer tables.

## Test 1: Row Count Validation
Verify that data has been loaded into all Gold layer tables

In [0]:
SELECT 
    'DimCustomer' as TableName,
    COUNT(*) as RowCount
FROM gold_catalog.retail_gold.dimcustomer

UNION ALL

SELECT 
    'DimProduct' as TableName,
    COUNT(*) as RowCount
FROM gold_catalog.retail_gold.DimProduct

UNION ALL

SELECT 
    'DimStore' as TableName,
    COUNT(*) as RowCount
FROM gold_catalog.retail_gold.DimStore

UNION ALL

SELECT 
    'FactSales' as TableName,
    COUNT(*) as RowCount
FROM gold_catalog.retail_gold.FactSales

ORDER BY TableName;

## Test 2: SCD Type 2 Validation (DimCustomer)
Validate Slowly Changing Dimension Type 2 implementation

In [0]:
-- Each CustomerID should have exactly ONE active record (IsActive = 1)
SELECT 
    CustomerID,
    COUNT(*) as ActiveRecordCount,
    CASE 
        WHEN COUNT(*) = 1 THEN '✓ PASS'
        ELSE '✗ FAIL - Multiple active records'
    END as ValidationStatus
FROM gold_catalog.retail_gold.dimcustomer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING COUNT(*) > 1
ORDER BY ActiveRecordCount DESC;

In [0]:
-- Validate date ranges: StartDate should be < EndDate for expired records
SELECT 
    COUNT(*) as InvalidDateRanges,
    CASE 
        WHEN COUNT(*) = 0 THEN '✓ PASS - All date ranges are valid'
        ELSE CONCAT('✗ FAIL - ', COUNT(*), ' records have invalid date ranges')
    END as ValidationStatus
FROM gold_catalog.retail_gold.dimcustomer
WHERE IsActive = 0 
  AND (StartDate >= EndDate OR EndDate IS NULL);

In [0]:
-- Check for overlapping active periods for the same customer
SELECT 
    a.CustomerID,
    a.CustomerSK as SK1,
    b.CustomerSK as SK2,
    a.StartDate as Start1,
    a.EndDate as End1,
    b.StartDate as Start2,
    b.EndDate as End2,
    '✗ FAIL - Overlapping periods' as ValidationStatus
FROM gold_catalog.retail_gold.dimcustomer a
JOIN gold_catalog.retail_gold.dimcustomer b
    ON a.CustomerID = b.CustomerID
    AND a.CustomerSK < b.CustomerSK
WHERE a.StartDate < COALESCE(b.EndDate, CURRENT_DATE())
  AND b.StartDate < COALESCE(a.EndDate, CURRENT_DATE());

## Test 3: Data Quality Checks
Validate data completeness and integrity

In [0]:
SELECT 
    'DimCustomer' as TableName,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) as Null_CustomerID,
    SUM(CASE WHEN CustomerName IS NULL THEN 1 ELSE 0 END) as Null_CustomerName,
    SUM(CASE WHEN Email IS NULL THEN 1 ELSE 0 END) as Null_Email,
    CASE 
        WHEN SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
        ELSE '✗ FAIL - NULL CustomerIDs found'
    END as ValidationStatus
FROM gold_catalog.retail_gold.dimcustomer

UNION ALL

SELECT 
    'DimProduct' as TableName,
    SUM(CASE WHEN ProductID IS NULL THEN 1 ELSE 0 END) as Null_ProductID,
    SUM(CASE WHEN ProductName IS NULL THEN 1 ELSE 0 END) as Null_ProductName,
    SUM(CASE WHEN Category IS NULL THEN 1 ELSE 0 END) as Null_Category,
    CASE 
        WHEN SUM(CASE WHEN ProductID IS NULL THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
        ELSE '✗ FAIL - NULL ProductIDs found'
    END as ValidationStatus
FROM gold_catalog.retail_gold.DimProduct

UNION ALL

SELECT 
    'DimStore' as TableName,
    SUM(CASE WHEN StoreID IS NULL THEN 1 ELSE 0 END) as Null_StoreID,
    SUM(CASE WHEN StoreName IS NULL THEN 1 ELSE 0 END) as Null_StoreName,
    SUM(CASE WHEN Region IS NULL THEN 1 ELSE 0 END) as Null_Region,
    CASE 
        WHEN SUM(CASE WHEN StoreID IS NULL THEN 1 ELSE 0 END) = 0 THEN '✓ PASS'
        ELSE '✗ FAIL - NULL StoreIDs found'
    END as ValidationStatus
FROM gold_catalog.retail_gold.DimStore;

In [0]:
-- Check for duplicate active ProductIDs in DimProduct
SELECT 
    'DimProduct' as TableName,
    ProductID,
    COUNT(*) as DuplicateCount,
    '✗ FAIL - Duplicate ProductIDs' as ValidationStatus
FROM gold_catalog.retail_gold.DimProduct
GROUP BY ProductID
HAVING COUNT(*) > 1

UNION ALL

-- Check for duplicate active StoreIDs in DimStore
SELECT 
    'DimStore' as TableName,
    StoreID,
    COUNT(*) as DuplicateCount,
    '✗ FAIL - Duplicate StoreIDs' as ValidationStatus
FROM gold_catalog.retail_gold.DimStore
GROUP BY StoreID
HAVING COUNT(*) > 1;

## Test 4: Referential Integrity
Validate foreign key relationships between Fact and Dimension tables

In [0]:
SELECT 
    'Orphaned CustomerSK' as Issue,
    COUNT(DISTINCT f.CustomerSK) as OrphanedCount,
    CASE 
        WHEN COUNT(DISTINCT f.CustomerSK) = 0 THEN '✓ PASS'
        ELSE CONCAT('✗ FAIL - ', COUNT(DISTINCT f.CustomerSK), ' orphaned CustomerSK values')
    END as ValidationStatus
FROM gold_catalog.retail_gold.FactSales f
LEFT JOIN gold_catalog.retail_gold.dimcustomer c ON f.CustomerSK = c.CustomerSK
WHERE f.CustomerSK IS NOT NULL AND c.CustomerSK IS NULL

UNION ALL

SELECT 
    'Orphaned ProductSK' as Issue,
    COUNT(DISTINCT f.ProductSK) as OrphanedCount,
    CASE 
        WHEN COUNT(DISTINCT f.ProductSK) = 0 THEN '✓ PASS'
        ELSE CONCAT('✗ FAIL - ', COUNT(DISTINCT f.ProductSK), ' orphaned ProductSK values')
    END as ValidationStatus
FROM gold_catalog.retail_gold.FactSales f
LEFT JOIN gold_catalog.retail_gold.DimProduct p ON f.ProductSK = p.ProductSK
WHERE f.ProductSK IS NOT NULL AND p.ProductSK IS NULL

UNION ALL

SELECT 
    'Orphaned StoreSK' as Issue,
    COUNT(DISTINCT f.StoreSK) as OrphanedCount,
    CASE 
        WHEN COUNT(DISTINCT f.StoreSK) = 0 THEN '✓ PASS'
        ELSE CONCAT('✗ FAIL - ', COUNT(DISTINCT f.StoreSK), ' orphaned StoreSK values')
    END as ValidationStatus
FROM gold_catalog.retail_gold.FactSales f
LEFT JOIN gold_catalog.retail_gold.DimStore s ON f.StoreSK = s.StoreSK
WHERE f.StoreSK IS NOT NULL AND s.StoreSK IS NULL;

## Test 5: Business Rule Validation
Validate business logic and constraints

In [0]:
SELECT 
    COUNT(*) as TotalRecords,
    SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) as Invalid_Quantity,
    SUM(CASE WHEN Amount <= 0 THEN 1 ELSE 0 END) as Invalid_Amount,
    SUM(CASE WHEN TransactionID IS NULL THEN 1 ELSE 0 END) as Null_TransactionID,
    SUM(CASE WHEN TxnDate IS NULL THEN 1 ELSE 0 END) as Null_TxnDate,
    CASE 
        WHEN SUM(CASE WHEN Quantity <= 0 OR Amount <= 0 OR TransactionID IS NULL OR TxnDate IS NULL THEN 1 ELSE 0 END) = 0 
        THEN '✓ PASS - All business rules satisfied'
        ELSE '✗ FAIL - Business rule violations found'
    END as ValidationStatus
FROM gold_catalog.retail_gold.FactSales;

In [0]:
SELECT 
    COUNT(*) as TotalProducts,
    SUM(CASE WHEN UnitPrice < 0 THEN 1 ELSE 0 END) as Negative_Prices,
    SUM(CASE WHEN UnitPrice = 0 THEN 1 ELSE 0 END) as Zero_Prices,
    CASE 
        WHEN SUM(CASE WHEN UnitPrice < 0 THEN 1 ELSE 0 END) = 0 
        THEN '✓ PASS - All prices are valid'
        ELSE CONCAT('✗ FAIL - ', SUM(CASE WHEN UnitPrice < 0 THEN 1 ELSE 0 END), ' negative prices found')
    END as ValidationStatus
FROM gold_catalog.retail_gold.DimProduct;

## Test 6: Source to Target Reconciliation
Compare Gold layer with Silver layer to ensure completeness

In [0]:
SELECT 
    'Products' as Dimension,
    (SELECT COUNT(*) FROM silver_catalog.retail_silver.silver_products) as Silver_Count,
    (SELECT COUNT(*) FROM gold_catalog.retail_gold.DimProduct) as Gold_Count,
    CASE 
        WHEN (SELECT COUNT(*) FROM silver_catalog.retail_silver.silver_products) = 
             (SELECT COUNT(*) FROM gold_catalog.retail_gold.DimProduct)
        THEN '✓ PASS'
        ELSE '✗ FAIL'
    END as ValidationStatus

UNION ALL

SELECT 
    'Stores' as Dimension,
    (SELECT COUNT(*) FROM silver_catalog.retail_silver.silver_stores) as Silver_Count,
    (SELECT COUNT(*) FROM gold_catalog.retail_gold.DimStore) as Gold_Count,
    CASE 
        WHEN (SELECT COUNT(*) FROM silver_catalog.retail_silver.silver_stores) = 
             (SELECT COUNT(*) FROM gold_catalog.retail_gold.DimStore)
        THEN '✓ PASS'
        ELSE '✗ FAIL'
    END as ValidationStatus;

## Test 7: Validation Summary Report
Overall validation status across all tests

In [0]:
-- This summary combines all key validation checks
WITH ValidationChecks AS (
    SELECT 'Row Counts' as TestCategory, 
           CASE WHEN COUNT(*) > 0 THEN '✓ PASS' ELSE '✗ FAIL' END as Status
    FROM gold_catalog.retail_gold.FactSales
    
    UNION ALL
    
    SELECT 'SCD-2 Active Records' as TestCategory,
           CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END as Status
    FROM (
        SELECT CustomerID FROM gold_catalog.retail_gold.dimcustomer 
        WHERE IsActive = 1 
        GROUP BY CustomerID HAVING COUNT(*) > 1
    )
    
    UNION ALL
    
    SELECT 'NULL Key Columns' as TestCategory,
           CASE WHEN SUM(null_count) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END as Status
    FROM (
        SELECT COUNT(*) as null_count FROM gold_catalog.retail_gold.dimcustomer WHERE CustomerID IS NULL
        UNION ALL
        SELECT COUNT(*) FROM gold_catalog.retail_gold.DimProduct WHERE ProductID IS NULL
        UNION ALL
        SELECT COUNT(*) FROM gold_catalog.retail_gold.DimStore WHERE StoreID IS NULL
        UNION ALL
        SELECT COUNT(*) FROM gold_catalog.retail_gold.FactSales WHERE TransactionID IS NULL
    )
    
    UNION ALL
    
    SELECT 'Business Rules (Positive Values)' as TestCategory,
           CASE WHEN COUNT(*) = 0 THEN '✓ PASS' ELSE '✗ FAIL' END as Status
    FROM gold_catalog.retail_gold.FactSales
    WHERE Quantity <= 0 OR Amount <= 0
)

SELECT 
    TestCategory,
    Status,
    CASE 
        WHEN Status = '✓ PASS' THEN 'All validations passed'
        ELSE 'Review detailed test results above'
    END as Recommendation
FROM ValidationChecks
ORDER BY 
    CASE WHEN Status = '✗ FAIL' THEN 1 ELSE 2 END,
    TestCategory;

In [0]:
-- Verify each CustomerID has only one unique name and one active record
SELECT 
    COUNT(DISTINCT CustomerID) as Total_Unique_Customers,
    COUNT(*) as Total_Records,
    SUM(CASE WHEN IsActive = 1 THEN 1 ELSE 0 END) as Active_Records,
    SUM(CASE WHEN IsActive = 0 THEN 1 ELSE 0 END) as Inactive_Records,
    COUNT(DISTINCT CONCAT(CustomerID, '|', CustomerName)) as Unique_CustomerID_Name_Combinations,
    CASE 
        WHEN COUNT(DISTINCT CustomerID) = COUNT(DISTINCT CONCAT(CustomerID, '|', CustomerName))
        THEN '✓ PASS - Each CustomerID has exactly one name'
        ELSE '✗ FAIL - Multiple names found for some CustomerIDs'
    END as Name_Consistency_Check
FROM gold_catalog.retail_gold.dimcustomer;